# Atelier ML - Apprentissage: Partie 3 (Entrainement)

## Phase d'apprentissage

But: determiner si un Pokemone a de grandes chances de gagner

Nous disposons de données d'apprentissage : nous sommes donc dans un cas de Machine Learning **supervisé**. Il existe deux grandes familles d'algorithmes :
- la **classification**, qui organise les prédictions en groupes
- la **régression**, qui prédit une valeur

Il faut predir le pourcentage de victoire, donc une valeur. Pour cela, il faut utiliser les algoritmes de regressions:
1.  La régression linéaire
2.  Les arbres de décisions
3.  Les forêts aléatoires

### Premiere etape
La première étape avant tout apprentissage est de découper les observations dont nous disposons en un jeu d’apprentissage avec lequel la machine va réaliser son apprentissage et en jeu de tests avec lequel la machine va évaluer son apprentissage. Pour cela, nous allons utiliser le module Scikit-Learn que nous vous invitons à ajouter à votre projet (avec les packages).

In [1]:
import pandas as pnd

#Chargement du dataset avec pour séparateur des tabulations
dataset = pnd.read_csv('data/dataset.csv', delimiter='\t')

print("Avant dropna :", dataset.shape)

#Suppression des lignes ayant des valeurs manquantes
dataset = dataset.dropna(axis=0, how='any')

print("Après dropna :", dataset.shape)

Avant dropna : (800, 18)
Après dropna : (404, 18)


## Découpage en jeu d'apprentissage et jeu de tests

### Variables prédictives (X) et variable à prédire (y)
- **X**: `POINTS_ATTAQUE`, `POINTS_DEFFENCE`, `POINTS_ATTAQUE_SPECIALE`, `POINT_DEFENSE_SPECIALE`, `POINTS_VITESSE`, `NOMBRE_GENERATIONS`
- **Y**: `POURCENTAGE_DE_VICTOIRE`

Chaque variable sera ensuite découpée en un groupe d’apprentissage (Train) et en un groupe de test (Test). Nous aurons donc un groupe `X_APPRENTISSAGE`, `Y_APPRENTISSAGE` permettant à la machine d’apprendre et un groupe `X_VALIDATION`,`Y_VALIDATION` lui permettant de valider son apprentissage

In [2]:
#X = on prend toutes les données, mais uniquement les features 4 à 11
# POINTS_ATTAQUE;POINTS_DEFFENCE;POINTS_ATTAQUE_SPECIALE;POINT_DEFENSE_SPECIALE;POINTS_VITESSE;NOMBRE_GENERATIONS
X = dataset.iloc[:, 5:12].values

#y = on prend uniquement la colonne POURCENTAGE_DE_VICTOIRE (16ème feature) 
#les : signifiant "Pour toutes les observations"
y = dataset.iloc[:, 16].values

#Construction du jeu d'entrainement et du jeu de tests
from sklearn.model_selection import train_test_split
X_APPRENTISSAGE, X_VALIDATION, Y_APPRENTISSAGE, Y_VALIDATION = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Algorithme de régression linéaire

Dans notre cas, nous avons plusieurs variables prédictives pour une prédiction. Nous sommes donc dans le cas d’une régression linéaire multiple. Voici comment utiliser l’algorithme de régression linéaire pour réaliser notre apprentissage à l’aide du module Scikit-Learn :

In [3]:
#---- ALGORITHME 1: REGRESSION LINEAIRE -----
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

#Choix de l'algorithme
algorithme = LinearRegression()

#Apprentissage à l'aide de la fonction fit
#Apprentisage effectué sur le jeu de données d'apprentissage
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)

#Realisation de prédictions sur le jeu de tests (validation)
predictions = algorithme.predict(X_VALIDATION)

#Calcul de la précision de l'apprentissage à l'aide de la
#fonction r2_score en comparant les valeurs prédites
#(predictions) et les valeurs attendues (Y_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- REGRESSION LINEAIRE -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- REGRESSION LINEAIRE -----------
>> Precision = 0.8828844617689129
------------------------------------------


La précision est mesurée avec `r2_score` sur les données de **tests**, que l'algorithme ne connaît pas. On la mesure aussi sur les données d'apprentissage : un grand écart entre les deux indique un **surapprentissage**

## L’arbre de décision appliqué à la régression
Passons à présent à l’utilisation des arbres de décisions en faisant attention à bien utiliser l’algorithme destiné aux problèmes de régression : `DecisionTreeRegressor` et non celui destiné aux problèmes de classification : `DecisionTreeClassifier`

In [4]:
#Choix de l'algorithme
from sklearn.tree import DecisionTreeRegressor
algorithme = DecisionTreeRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- ARBRES DE DECISIONS -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- ARBRES DE DECISIONS -----------
>> Precision = 0.7686239031358412
------------------------------------------


## La random forest
Toujours appliquée à la régression : `RandomForestRegressor`

In [5]:
#Choix de l'algorithme
from sklearn.ensemble import RandomForestRegressor
algorithme = RandomForestRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- FORETS ALEATOIRES -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- FORETS ALEATOIRES -----------
>> Precision = 0.897337213701789
------------------------------------------


**À propos du surapprentissage (overfitting) :** Le surapprentissage (overfitting) est un phénomène se traduisant par le fait que la solution est trop adaptée aux données d’apprentissage et ne se généralise pas à de nouvelles données qui lui sont inconnues. Ainsi, si pour un algorithme nous obtenons une précision 99 % sur les données d’apprentissage et que nous obtenons une valeur de 20 % sur les données de tests, il y fort à parier que nous sommes en présence d’un surapprentissage.

**Le sous-apprentissage (underfitting)**: ce dernier se produit lorsque l’algorithme n’arrive pas à trouver une corrélation entre les données d’apprentissage et n’arrive donc pas à réaliser de bonnes prédictions. La notion d’overfitting est souvent rencontrée lors de l’utilisation des réseaux de neurones. En effet, lors de l’apprentissage nous allons chercher à minimiser les erreurs de prédictions en réalisant un certain nombre de boucles d’apprentissage (itérations) où, à chaque boucle, l’algorithme va apprendre de ses erreurs et se corriger. Plus nous réalisons d’itérations, plus l’algorithme apprend et moins il se trompe. Si trop d’itérations sont réalisées, l’algorithme aura une très bonne précision sur les données d’apprentissage, mais il se sera spécialisé sur celles-ci et aura une mauvaise précision sur les données de tests. La généralisation n’est donc plus possible

## Sauvegarde du modèle d’apprentissage

In [6]:
import joblib
fichier = 'modele/modele_pokemon.mod'
joblib.dump(algorithme, fichier)

['modele/modele_pokemon.mod']